In [1]:

# 05 : Step 1: session bootstrap
import sys, os, importlib

SCRIPTS = "/content/drive/MyDrive/0_potato_project_v1/scripts"

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path = [p for p in sys.path if p != SCRIPTS]
sys.path.insert(0, SCRIPTS)
importlib.invalidate_caches()          # Python caches "nothing here" from failed lookups

import torch
import config as C, data as D, model as M

D.setup_data()                         # idempotent restore of /content/potato

print("config  :", C.PROJECT.name)
print("cuda    :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")
print("modules :", C.__name__, D.__name__, M.__name__)

Mounted at /content/drive
restoring /content/drive/MyDrive/0_potato_project_v1/data/potato_raw/raw -> /content/potato
  Early_blight -> Potato___Early_blight
  Late_blight -> Potato___Late_blight
  Healthy -> Potato___healthy
config  : 0_potato_project_v1
cuda    : True | Tesla T4
modules : config data model


In [2]:

# 05: Step 2: constants + seeding
import random
import numpy as np
import torch

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP     = DEVICE.type == "cuda"
HEALTHY = C.CLASS_TO_IDX["Potato___healthy"]


def set_seed(seed=C.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()

print("device        :", DEVICE)
print("amp enabled   :", AMP)
print("healthy index :", HEALTHY, "->", C.IDX_TO_CLASS[HEALTHY])
print("seed          :", C.SEED)
print("targets       : macro-F1 >=", C.TARGET_MACRO_F1,
      "| healthy recall >=", C.TARGET_HEALTHY_RECALL)

device        : cuda
amp enabled   : True
healthy index : 2 -> Potato___healthy
seed          : 42
targets       : macro-F1 >= 0.95 | healthy recall >= 0.9


In [3]:

# 05: Step 3: one training epoch
def train_one_epoch(model, loader, criterion, optimizer, scaler=None):
    """One pass over `loader` with weight updates. Returns mean training loss."""
    model.train()                                   # BN guard fires here
    total, n = 0.0, 0

    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", enabled=AMP):
            loss = criterion(model(xb), yb)

        if scaler is not None:
            scaler.scale(loss).backward()           # scale up: avoid fp16 underflow
            scaler.step(optimizer)                  # unscales, then steps
            scaler.update()                         # adapt the scale factor
        else:
            loss.backward()
            optimizer.step()

        total += loss.item() * yb.size(0)
        n += yb.size(0)

    return total / n


print("train_one_epoch defined")
print("signature:", train_one_epoch.__code__.co_varnames[:5])

train_one_epoch defined
signature: ('model', 'loader', 'criterion', 'optimizer', 'scaler')


In [4]:

# 05 : Step 4: evaluation pass
from sklearn.metrics import f1_score, recall_score


@torch.no_grad()
def evaluate(model, loader, criterion):
    """No-update pass. Returns (metrics, probs, ys).

    Loader order is fixed (shuffle=False), so probs rows align with the
    loader's index list — that alignment is what makes OOF predictions joinable.
    """
    model.eval()
    total, n, P, Y = 0.0, 0, [], []

    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        with torch.autocast("cuda", enabled=AMP):
            logits = model(xb)
            loss = criterion(logits, yb)

        total += loss.item() * yb.size(0)
        n += yb.size(0)
        P.append(torch.softmax(logits.float(), dim=1).cpu())   # back to fp32
        Y.append(yb.cpu())

    probs = torch.cat(P).numpy()
    ys = torch.cat(Y).numpy()
    preds = probs.argmax(1)

    metrics = {
        "loss":           total / n,
        "acc":            float((preds == ys).mean()),
        "macro_f1":       f1_score(ys, preds, average="macro", zero_division=0),
        "healthy_recall": recall_score(ys, preds, labels=[HEALTHY],
                                       average="macro", zero_division=0),
    }
    return metrics, probs, ys


print("evaluate defined")
print("returns  : metrics dict, probs (n,3), ys (n,)")
print("metrics  : loss, acc, macro_f1, healthy_recall")

evaluate defined
returns  : metrics dict, probs (n,3), ys (n,)
metrics  : loss, acc, macro_f1, healthy_recall


In [5]:

# 05: Step 5: assemble fold 0 (smoke sizes)
import torch.nn as nn

set_seed()

df = D.load_manifest()
tl, vl, el, w = D.get_loaders(df, fold=0, smoke=True)

net    = M.set_stage(M.build_model(), stage=1).to(DEVICE)
crit   = nn.CrossEntropyLoss(weight=w.to(DEVICE))
opt    = torch.optim.AdamW(M.param_groups(net, stage=1), weight_decay=1e-4)
scaler = torch.amp.GradScaler(enabled=AMP)

tr_p, fz_p = M.count_params(net)
print(f"manifest   : {df.shape}")
print(f"batches    : train {len(tl)}  val {len(vl)}  eval {len(el)}  (smoke)")
print(f"weights    : {[round(v, 3) for v in w.tolist()]}")
print(f"stage 1    : trainable {tr_p:,}  frozen {fz_p:,}")
print(f"optimizer  : {[(g['name'], g['lr']) for g in opt.param_groups]}")
print(f"model on   : {next(net.parameters()).device}")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 148MB/s]


manifest   : (2152, 13)
batches    : train 2  val 1  eval 1  (smoke)
weights    : [0.821, 0.667, 3.556]
stage 1    : trainable 1,233,923  frozen 2,971,952
optimizer  : [('head', 0.001)]
model on   : cuda:0


In [6]:

# 05 : Step 6: one smoke epoch on fold 0
import time
import numpy as np

before, _, _ = evaluate(net, vl, crit)

t0 = time.time()
tr_loss = train_one_epoch(net, tl, crit, opt, scaler)
elapsed = time.time() - t0

after, probs, ys = evaluate(net, vl, crit)

print(f"train loss     : {tr_loss:.4f}   ({elapsed:.1f}s, 2 batches)")
print(f"val loss       : {before['loss']:.4f} -> {after['loss']:.4f}   "
      f"({'down' if after['loss'] < before['loss'] else 'UP'})")
print(f"val acc        : {before['acc']:.3f} -> {after['acc']:.3f}")
print(f"val macro-F1   : {before['macro_f1']:.3f} -> {after['macro_f1']:.3f}")
print(f"probs          : {probs.shape}   rows sum to 1: {np.allclose(probs.sum(1), 1)}")
print(f"labels in val  : {np.bincount(ys, minlength=3).tolist()}")
print(f"grad scale     : {scaler.get_scale():.0f}")

train loss     : 1.2009   (0.8s, 2 batches)
val loss       : 1.0583 -> 0.8701   (down)
val acc        : 0.562 -> 0.312
val macro-F1   : 0.392 -> 0.298
probs          : (32, 3)   rows sum to 1: True
labels in val  : [14, 14, 4]
grad scale     : 32768


In [7]:

# 05 : Step 7: run one stage with early stopping
import copy


def run_stage(model, train_loader, val_loader, criterion, optimizer, scaler,
              epochs, stage, patience=C.EARLY_STOP_PATIENCE, fold=None, verbose=True):
    """Train until val loss stops improving. Returns (history, best_epoch, best_val).

    The model is left holding the BEST weights seen, not the last ones.
    """
    best_loss, best_state, best_epoch, bad = float("inf"), None, -1, 0
    history = []

    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val, _, _ = evaluate(model, val_loader, criterion)

        improved = val["loss"] < best_loss
        if improved:
            best_loss, best_epoch, bad = val["loss"], ep, 0
            best_state = copy.deepcopy(
                {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        else:
            bad += 1

        history.append({"fold": fold, "stage": stage, "epoch": ep,
                        "train_loss": tr_loss, **{f"val_{k}": v for k, v in val.items()},
                        "lr": optimizer.param_groups[0]["lr"],
                        "secs": round(time.time() - t0, 1)})

        if verbose:
            print(f"  s{stage} e{ep:>2}  train {tr_loss:.4f}  val {val['loss']:.4f}"
                  f"  F1 {val['macro_f1']:.3f}  healthy-R {val['healthy_recall']:.3f}"
                  f"  {time.time()-t0:.0f}s{'  *' if improved else ''}")

        if bad >= patience:
            if verbose:
                print(f"  early stop: no improvement in {patience} epochs")
            break

    if best_state is not None:
        model.load_state_dict(best_state)          # restore the peak, not the last
    return history, best_epoch, best_loss


print("run_stage defined")
print("stops after :", C.EARLY_STOP_PATIENCE, "epochs without improvement in",
      C.EARLY_STOP_MONITOR)
print("epochs      : stage 1 =", C.EPOCHS_HEAD, " stage 2 =", C.EPOCHS_TUNE)

run_stage defined
stops after : 4 epochs without improvement in val_loss
epochs      : stage 1 = 10  stage 2 = 15


In [8]:

# 05: Step 8a: run directory
from datetime import datetime
from pathlib import Path

RUN_TAG = f"{datetime.now():%Y%m%d_%H%M}_{C.AUG_MODE}"
RUN_DIR = C.RESULTS / "runs" / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = RUN_DIR / "summary.csv"


def fold_dir(fold, run_dir=None):
    """Per-fold folder inside the current run. Created on demand."""
    d = (run_dir or RUN_DIR) / f"fold_{fold}"
    d.mkdir(parents=True, exist_ok=True)
    return d


def fold_done(fold, run_dir=None):
    """True if this fold already has all three artifacts — used to skip on resume."""
    d = (run_dir or RUN_DIR) / f"fold_{fold}"
    return all((d / f).exists() for f in ("best.pt", "history.csv", "oof_preds.csv"))


print("run tag   :", RUN_TAG)
print("run dir   :", RUN_DIR)
print("exists    :", RUN_DIR.exists())
print("fold 0 -> :", fold_dir(0).relative_to(C.PROJECT))
print("fold 0 done:", fold_done(0))
print("aug mode  :", C.AUG_MODE, "| balance:", C.BALANCE)

run tag   : 20260824_0822_baseline
run dir   : /content/drive/MyDrive/0_potato_project_v1/results/runs/20260824_0822_baseline
exists    : True
fold 0 -> : results/runs/20260824_0822_baseline/fold_0
fold 0 done: False
aug mode  : baseline | balance: loss


In [ ]:

# 05:Cell 8b: train one complete fold
import pandas as pd

def run_fold(df, fold, run_dir=None, smoke=False, verbose=True):
    """Stage 1 -> stage 2 -> one held-out evaluation. Writes 3 files. Returns metrics."""
    d = fold_dir(fold, run_dir)
    set_seed(C.SEED + fold)

    tl, vl, el, w = D.get_loaders(df, fold=fold, smoke=smoke)
    net = M.set_stage(M.build_model(), stage=1).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=w.to(DEVICE)) if w is not None else nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler(enabled=AMP)

    if verbose:
        print(f"\n── fold {fold} ──  train {len(tl.dataset)}  "
              f"val {len(vl.dataset)}  eval {len(el.dataset)}")

    # stage 1: head only
    opt = torch.optim.AdamW(M.param_groups(net, stage=1), weight_decay=1e-4)
    h1, be1, bl1 = run_stage(net, tl, vl, crit, opt, scaler,
                             epochs=1 if smoke else C.EPOCHS_HEAD,
                             stage=1, fold=fold, verbose=verbose)

    # stage 2: unfreeze late blocks
    M.set_stage(net, stage=2)
    opt = torch.optim.AdamW(M.param_groups(net, stage=2), weight_decay=1e-4)
    h2, be2, bl2 = run_stage(net, tl, vl, crit, opt, scaler,
                             epochs=1 if smoke else C.EPOCHS_TUNE,
                             stage=2, fold=fold, verbose=verbose)

    # held-out fold: first and only look
    test, probs, ys = evaluate(net, el, crit)

    # artifacts
    pd.DataFrame(h1 + h2).to_csv(d / "history.csv", index=False)

    idx = D.get_indices(df, fold)[2]
    pd.DataFrame({
        "path": df.path.values[idx], "y_true": ys,
        **{f"p{i}": probs[:, i] for i in range(C.NUM_CLASSES)},
        "y_pred": probs.argmax(1), "confidence": probs.max(1), "fold": fold,
    }).to_csv(d / "oof_preds.csv", index=False)

    M.save_checkpoint(d / "best.pt", net, fold=fold, stage=2,
                      epoch=be2, metrics=test)

    row = {"fold": fold, **{f"test_{k}": v for k, v in test.items()},
           "best_ep_s1": be1, "best_val_s1": bl1,
           "best_ep_s2": be2, "best_val_s2": bl2,
           "n_train": len(tl.dataset), "n_eval": len(el.dataset)}

    if verbose:
        print(f"  HELD-OUT  F1 {test['macro_f1']:.4f}  "
              f"healthy-R {test['healthy_recall']:.4f}  acc {test['acc']:.4f}")
    return row


print("run_fold defined")
print("writes    : best.pt, history.csv, oof_preds.csv")
print("per fold  : stage 1 <=", C.EPOCHS_HEAD, "ep, stage 2 <=", C.EPOCHS_TUNE, "ep")

run_fold defined
writes    : best.pt, history.csv, oof_preds.csv
per fold  : stage 1 <= 10 ep, stage 2 <= 15 ep


In [10]:

# 05 : Step 8b-fix: take paths from the loader
def run_fold(df, fold, run_dir=None, smoke=False, verbose=True):
    """Stage 1 -> stage 2 -> one held-out evaluation. Writes 3 files. Returns metrics."""
    d = fold_dir(fold, run_dir)
    set_seed(C.SEED + fold)

    tl, vl, el, w = D.get_loaders(df, fold=fold, smoke=smoke)
    net = M.set_stage(M.build_model(), stage=1).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=w.to(DEVICE)) if w is not None else nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler(enabled=AMP)

    if verbose:
        print(f"\n── fold {fold} ──  train {len(tl.dataset)}  "
              f"val {len(vl.dataset)}  eval {len(el.dataset)}")

    opt = torch.optim.AdamW(M.param_groups(net, stage=1), weight_decay=1e-4)
    h1, be1, bl1 = run_stage(net, tl, vl, crit, opt, scaler,
                             epochs=1 if smoke else C.EPOCHS_HEAD,
                             stage=1, fold=fold, verbose=verbose)

    M.set_stage(net, stage=2)
    opt = torch.optim.AdamW(M.param_groups(net, stage=2), weight_decay=1e-4)
    h2, be2, bl2 = run_stage(net, tl, vl, crit, opt, scaler,
                             epochs=1 if smoke else C.EPOCHS_TUNE,
                             stage=2, fold=fold, verbose=verbose)

    test, probs, ys = evaluate(net, el, crit)

    pd.DataFrame(h1 + h2).to_csv(d / "history.csv", index=False)

    # paths come from the loader's own dataset — same order, same subsampling,
    # no re-derivation that could drift out of sync with what was evaluated
    paths = el.dataset.paths
    assert len(paths) == len(ys), f"path/label mismatch: {len(paths)} vs {len(ys)}"
    assert (el.dataset.ys == ys).all(), "loader order changed between build and eval"

    pd.DataFrame({
        "path": paths, "y_true": ys,
        **{f"p{i}": probs[:, i] for i in range(C.NUM_CLASSES)},
        "y_pred": probs.argmax(1), "confidence": probs.max(1), "fold": fold,
    }).to_csv(d / "oof_preds.csv", index=False)

    M.save_checkpoint(d / "best.pt", net, fold=fold, stage=2,
                      epoch=be2, metrics=test)

    row = {"fold": fold, **{f"test_{k}": v for k, v in test.items()},
           "best_ep_s1": be1, "best_val_s1": bl1,
           "best_ep_s2": be2, "best_val_s2": bl2,
           "n_train": len(tl.dataset), "n_eval": len(el.dataset)}

    if verbose:
        print(f"  HELD-OUT  F1 {test['macro_f1']:.4f}  "
              f"healthy-R {test['healthy_recall']:.4f}  acc {test['acc']:.4f}")
    return row


print("run_fold redefined — paths sourced from el.dataset")

run_fold redefined — paths sourced from el.dataset


In [11]:

# 05 : Step 8c: smoke run_fold (throwaway output)
SMOKE_DIR = C.RESULTS / "runs" / "_smoke"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

df = D.load_manifest()
row = run_fold(df, fold=0, run_dir=SMOKE_DIR, smoke=True)

d = SMOKE_DIR / "fold_0"
print("\nfiles written:")
for f in sorted(d.iterdir()):
    print(f"  {f.name:<16} {f.stat().st_size/1024:>8.1f} KB")

oof = pd.read_csv(d / "oof_preds.csv")
hist = pd.read_csv(d / "history.csv")
print("\noof_preds :", oof.shape, "| cols:", list(oof.columns))
print("probs sum :", np.allclose(oof[["p0", "p1", "p2"]].sum(1), 1))
print("paths uniq:", oof.path.nunique() == len(oof))
print("history   :", hist.shape, "| stages:", sorted(hist.stage.unique()))
print("\nsummary row keys:", list(row.keys()))


── fold 0 ──  train 64  val 32  eval 32
  s1 e 1  train 1.2009  val 0.8701  F1 0.298  healthy-R 1.000  1s  *
  s2 e 1  train 0.8868  val 0.8016  F1 0.341  healthy-R 1.000  3s  *
  HELD-OUT  F1 0.3300  healthy-R 1.0000  acc 0.3438

files written:
  best.pt           16618.4 KB
  history.csv           0.2 KB
  oof_preds.csv         4.0 KB

oof_preds : (32, 8) | cols: ['path', 'y_true', 'p0', 'p1', 'p2', 'y_pred', 'confidence', 'fold']
probs sum : True
paths uniq: True
history   : (2, 10) | stages: [np.int64(1), np.int64(2)]

summary row keys: ['fold', 'test_loss', 'test_acc', 'test_macro_f1', 'test_healthy_recall', 'best_ep_s1', 'best_val_s1', 'best_ep_s2', 'best_val_s2', 'n_train', 'n_eval']


In [21]:
import shutil
shutil.rmtree(SMOKE_DIR)
print("smoke removed:", not SMOKE_DIR.exists())
print("run dir clean:", sorted(p.name for p in RUN_DIR.iterdir()) or "empty")

smoke removed: True
run dir clean: ['fold_0']


In [12]:

# 05: Step 9a: fold 0, full run
t0 = time.time()

df = D.load_manifest()
row0 = run_fold(df, fold=0, smoke=False)

pd.DataFrame([row0]).to_csv(SUMMARY_CSV, index=False)

print(f"\ntotal        : {(time.time()-t0)/60:.1f} min")
print(f"summary      : {SUMMARY_CSV.relative_to(C.PROJECT)}")
print(f"macro-F1     : {row0['test_macro_f1']:.4f}  (target {C.TARGET_MACRO_F1})")
print(f"healthy-R    : {row0['test_healthy_recall']:.4f}  (target {C.TARGET_HEALTHY_RECALL})")
print(f"best epochs  : stage 1 = {row0['best_ep_s1']}  stage 2 = {row0['best_ep_s2']}")


── fold 0 ──  train 1477  val 246  eval 429
  s1 e 1  train 0.1854  val 0.0315  F1 0.984  healthy-R 0.947  21s  *
  s1 e 2  train 0.0624  val 0.0256  F1 0.974  healthy-R 1.000  12s  *
  s1 e 3  train 0.0665  val 0.0167  F1 0.990  healthy-R 1.000  12s  *
  s1 e 4  train 0.0318  val 0.0129  F1 0.987  healthy-R 1.000  13s  *
  s1 e 5  train 0.0294  val 0.0371  F1 0.978  healthy-R 1.000  13s
  s1 e 6  train 0.0425  val 0.0046  F1 1.000  healthy-R 1.000  13s  *
  s1 e 7  train 0.0170  val 0.0153  F1 0.990  healthy-R 0.947  13s
  s1 e 8  train 0.0412  val 0.0625  F1 0.976  healthy-R 1.000  13s
  s1 e 9  train 0.0277  val 0.0088  F1 0.997  healthy-R 1.000  12s
  s1 e10  train 0.0083  val 0.0194  F1 0.980  healthy-R 1.000  12s
  early stop: no improvement in 4 epochs
  s2 e 1  train 0.1054  val 0.0067  F1 1.000  healthy-R 1.000  14s  *
  s2 e 2  train 0.0567  val 0.0113  F1 0.990  healthy-R 1.000  13s
  s2 e 3  train 0.0450  val 0.0170  F1 0.980  healthy-R 1.000  13s
  s2 e 4  train 0.0420  v

In [13]:
# 05 : Step 9b: remaining folds, resumable
t0 = time.time()
df = D.load_manifest()

rows = pd.read_csv(SUMMARY_CSV).to_dict("records") if SUMMARY_CSV.exists() else []
done = {r["fold"] for r in rows}

for k in range(C.N_FOLDS):
    if k in done and fold_done(k):
        print(f"fold {k}: already complete — skipping")
        continue
    rows = [r for r in rows if r["fold"] != k]        # drop any partial row
    rows.append(run_fold(df, fold=k, smoke=False))
    rows.sort(key=lambda r: r["fold"])
    pd.DataFrame(rows).to_csv(SUMMARY_CSV, index=False)   # checkpoint each fold

s = pd.DataFrame(rows)
print(f"\ntotal: {(time.time()-t0)/60:.1f} min   folds: {len(s)}")
print(s[["fold", "test_macro_f1", "test_healthy_recall", "test_acc",
         "best_ep_s1", "best_ep_s2"]].to_string(index=False))

for m, target in [("test_macro_f1", C.TARGET_MACRO_F1),
                  ("test_healthy_recall", C.TARGET_HEALTHY_RECALL)]:
    mu, sd = s[m].mean(), s[m].std()
    print(f"\n{m:<20} {mu:.4f} ± {sd:.4f}   target {target}   "
          f"{'PASS' if mu >= target else 'MISS'}   min fold {s[m].min():.4f}")

fold 0: already complete — skipping

── fold 1 ──  train 1474  val 246  eval 432
  s1 e 1  train 0.2219  val 0.0757  F1 0.944  healthy-R 1.000  12s  *
  s1 e 2  train 0.0571  val 0.0912  F1 0.919  healthy-R 1.000  13s
  s1 e 3  train 0.0452  val 0.1777  F1 0.872  healthy-R 1.000  13s
  s1 e 4  train 0.0725  val 0.0491  F1 0.960  healthy-R 1.000  13s  *
  s1 e 5  train 0.0224  val 0.0287  F1 0.976  healthy-R 1.000  13s  *
  s1 e 6  train 0.0466  val 0.0659  F1 0.957  healthy-R 1.000  13s
  s1 e 7  train 0.0281  val 0.1599  F1 0.889  healthy-R 1.000  12s
  s1 e 8  train 0.0418  val 0.0376  F1 0.977  healthy-R 0.941  12s
  s1 e 9  train 0.0100  val 0.0217  F1 0.977  healthy-R 0.941  14s  *
  s1 e10  train 0.0250  val 0.0141  F1 0.994  healthy-R 1.000  14s  *
  s2 e 1  train 0.0710  val 0.0376  F1 0.960  healthy-R 1.000  14s  *
  s2 e 2  train 0.0521  val 0.0307  F1 0.966  healthy-R 1.000  14s  *
  s2 e 3  train 0.0458  val 0.0183  F1 0.979  healthy-R 1.000  14s  *
  s2 e 4  train 0.0427  